# リソースクリーンアップ

作成したリソースを削除します。

> **注意**: このnotebookを実行すると、作成した全てのリソースが削除されます。

In [ ]:
%pip install -U -qqqq databricks-vectorsearch databricks-sdk
dbutils.library.restartPython()

In [ ]:
%run ./_config

## Step 1: Vector Search Indexの削除

In [ ]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

vs_index_name = f"{catalog}.{schema}.knowledge_base_vs_index"
try:
    vsc.delete_index(VECTOR_SEARCH_ENDPOINT_NAME, vs_index_name)
    print(f"✅ Vector Search Index '{vs_index_name}' を削除しました")
except Exception as e:
    print(f"⚠️ Index削除スキップ: {e}")

## Step 2: Vector Search Endpointの削除

In [ ]:
try:
    vsc.delete_endpoint(VECTOR_SEARCH_ENDPOINT_NAME)
    print(f"✅ Vector Search Endpoint '{VECTOR_SEARCH_ENDPOINT_NAME}' を削除しました")
except Exception as e:
    print(f"⚠️ Endpoint削除スキップ: {e}")

## Step 3: UC関数の削除

In [ ]:
functions_to_drop = [
    "get_customer_by_email",
    "get_order_history",
    "get_support_tickets",
    "search_policy",
    "calculate_math_expression",
]

for func_name in functions_to_drop:
    try:
        spark.sql(f"DROP FUNCTION IF EXISTS {catalog}.{schema}.{func_name}")
        print(f"✅ 関数 '{func_name}' を削除しました")
    except Exception as e:
        print(f"⚠️ 関数 '{func_name}' 削除スキップ: {e}")

## Step 4: スキーマの削除

In [ ]:
try:
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE")
    print(f"✅ スキーマ '{catalog}.{schema}' を削除しました")
except Exception as e:
    print(f"⚠️ スキーマ削除スキップ: {e}")

## Step 5: MLflow Experimentの削除

In [ ]:
import mlflow
mlflow.set_tracking_uri("databricks")

# config.yaml の mlflow_experiment_name で作成したExperimentを削除
user_email = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
experiment_name = MLFLOW_EXPERIMENT_NAME
if not experiment_name.startswith("/"):
    experiment_name = f"/Users/{user_email}/{experiment_name}"

try:
    exp = mlflow.get_experiment_by_name(experiment_name)
    if exp and exp.lifecycle_stage != 'deleted':
        mlflow.delete_experiment(exp.experiment_id)
        print(f"✅ Experiment '{experiment_name}' を削除しました")
    else:
        print(f"⚠️ Experiment '{experiment_name}' は存在しないか既に削除済み")
except Exception as e:
    print(f"⚠️ Experiment削除スキップ: {e}")

## Step 6: 観測性スキーマの削除（オプション）

Delta Sync / OTEL用に作成したスキーマを削除します。  
他のExperimentと共有している場合はスキップしてください。

In [ ]:
# 観測性用スキーマの削除
obs_schema_full = f"{catalog}.{OBS_SCHEMA}"
try:
    spark.sql(f"DROP SCHEMA IF EXISTS {obs_schema_full} CASCADE")
    print(f"✅ 観測性スキーマ '{obs_schema_full}' を削除しました")
except Exception as e:
    print(f"⚠️ 観測性スキーマ削除スキップ: {e}")

## クリーンアップ完了

全てのリソースが削除されました。

> **注意**: AI Gatewayと埋め込みモデルは共有リソースのため削除していません。  
> Dify側のアプリ・設定は手動で削除してください。